# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access basic metadata fields
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}\n")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced via their `@id` values as per Croissant specification.

**Available record sets:**

In [ ]:
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
print("Record sets (by @id):")
for rs in dataset.metadata.recordSet:
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")

# Display fields present in each record set
for rs in dataset.metadata.recordSet:
    print(f"\nFields for record set {rs['@id']}:")
    for field in rs['field']:
        field_name = field.get('name', '(no name)') if isinstance(field, dict) else str(field)
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
        print(f"  - Field @id: {field_id} / Name: {field_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use record set and field `@id`s as shown above.

In [ ]:
# Choose the first record set for demonstration
record_sets = record_set_ids
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from {record_set_id}.")
    else:
        print(f"No records found for {record_set_id}.")

# Choose a populated record set for further exploration
for rid, df in dataframes.items():
    if not df.empty:
        primary_record_set = rid
        break

print(f"\nFields (columns) in record set {primary_record_set}:")
print(dataframes[primary_record_set].columns.tolist())
dataframes[primary_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data filtering and transformation steps. We'll use a numeric field from the record set (by its `@id`/column name).

In [ ]:
# List available fields to help the user identify a numeric one
df = dataframes[primary_record_set]
print("Column data types:")
print(df.dtypes)

# Choose a numeric field for demonstration. Replace below if a different field is numeric.
possible_numeric_fields = df.select_dtypes(include=['number']).columns.tolist()

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # Use the first numeric field
    print(f"\nUsing numeric field: {numeric_field}")
else:
    raise ValueError("No numeric fields found for EDA.")

# Filter records where numeric_field > threshold
threshold = df[numeric_field].mean()  # Use mean as example threshold
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by any available categorical field
possible_group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = None
for col in possible_group_fields:
    if df[col].nunique() < 20 and col != numeric_field:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo suitable grouping field found.")

## 5. Visualization
Visualize the distribution and relationships in the numeric field(s) using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (original and normalized)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f"Original distribution: {numeric_field}")

plt.subplot(1, 2, 2)
sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True, color='g')
plt.title(f"Normalized: {numeric_field}")
plt.tight_layout()
plt.show()

# If grouping field available, show barplot
if group_field:
    plt.figure(figsize=(8,4))
    group_mean = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    sns.barplot(data=group_mean, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:

- Loaded FAIR^2 dataset metadata and record sets using the Croissant schema and the `mlcroissant` library.
- Explored the provided record sets, their fields, and extracted sample data frames.
- Conducted filtering, normalization, and grouping on a numeric field.
- Visualized the distributions for both original and processed data attributes.

This workflow provides a reproducible approach for initial FAIR data exploration and quality assessment using the semantic structure defined by Croissant.